# Занятие 8. Голосовая биометрическая аутентификация
## Извлечение MFCC признаков и косинусное сопоставление

**Дисциплина:** Биометрические системы аутентификации  
**Выполнил:** студент гр. SIS-2309 Аканов Дархан (ID: 38041)

**Цель работы:** Реализовать базовую верификацию диктора по голосу с использованием MFCC-признаков и косинусного сходства.

## 1. Извлечение 13 мел-кепстральных коэффициентов (MFCC)

In [1]:
import os, sys, importlib
from pathlib import Path
from IPython.display import Image, display

# 1. Гарантированный поиск корня проекта без риска потери рабочей директории
curr = Path.cwd()
PROJECT_ROOT = curr if (curr / 'src').is_dir() else (curr.parent if (curr.parent / 'src').is_dir() else Path.cwd())
src_dir = PROJECT_ROOT / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# 2. Импорт и принудительная перезагрузка voice_authentication из диска (без необходимости рестарта Kernel)
import voice_authentication
importlib.reload(voice_authentication)

sample_path = PROJECT_ROOT / 'datasets' / 'voice_samples' / 'darkhan_akanov' / 'phrase_01.wav'
y, sr, mfcc, stft, mel, emb = voice_authentication.extract_embedding(sample_path)
print(f'Длина: {len(y)} отсчётов ({len(y)/sr:.2f} сек), MFCC shape: {mfcc.shape}, Embedding: {emb.shape}')

Длина: 40128 отсчётов (2.51 сек), MFCC shape: (20, 251), Embedding: (38,)


## 2. Запуск конвейера верификации голоса и расчёт EER

In [2]:
test_records, opt_th, eer = voice_authentication.run_voice_pipeline()
print(f'Готово! Оптимальный порог T* = {opt_th:.3f}, EER = {eer:.1f}%')

plot_path = PROJECT_ROOT / 'reports' / 'lab08_voice_features_analysis.png'
if plot_path.exists():
    display(Image(filename=str(plot_path)))

ЗАНЯТИЕ 8. ГОЛОСОВАЯ БИОМЕТРИЧЕСКАЯ АУТЕНТИФИКАЦИЯ (MFCC)
Студент: Аканов Дархан | Группа: SIS-2309 | ID: 38041

[ШАГ 1-2] Загрузка голосовых образцов и извлечение 13 MFCC:
  Субъект: Darkhan (Аканов Д.) — 5 образцов:
    - phrase_01.wav: 2.51 сек, фреймов: 251, MFCC размер: (20, 251)
    - phrase_02.wav: 3.50 сек, фреймов: 351, MFCC размер: (20, 351)
    - phrase_03.wav: 3.50 сек, фреймов: 351, MFCC размер: (20, 351)
    - phrase_04.wav: 2.51 сек, фреймов: 251, MFCC размер: (20, 251)
    - phrase_05.wav: 2.37 сек, фреймов: 237, MFCC размер: (20, 237)
  Субъект: Impostor 1 (Боб) — 5 образцов:
    - phrase_01.wav: 3.39 сек, фреймов: 340, MFCC размер: (20, 340)
    - phrase_02.wav: 3.39 сек, фреймов: 340, MFCC размер: (20, 340)
    - phrase_03.wav: 3.39 сек, фреймов: 340, MFCC размер: (20, 340)
    - phrase_04.wav: 3.39 сек, фреймов: 340, MFCC размер: (20, 340)
    - phrase_05.wav: 3.39 сек, фреймов: 340, MFCC размер: (20, 340)
  Субъект: Impostor 2 (Чарли) — 5 образцов:
    - phrase_01.

    - phrase_05.wav: 3.39 сек, фреймов: 340, MFCC размер: (20, 340)



[+] Сохранён график признаков: lab08_mfcc_features.png

[ШАГ 4-5] Тестирование верификации диктора (Cosine Similarity):
---------------------------------------------------------------------------
Образец                  | Истинный диктор      | Score    | Решение (T*=0.756)
---------------------------------------------------------------------------
Darkhan/фраза_4          | Дархан Аканов        | 0.9770  | ПРИНЯТ (Genuine)
Darkhan/фраза_5          | Дархан Аканов        | 0.9354  | ПРИНЯТ (Genuine)
Bob/фраза_1              | Боб (чужак)          | 0.5267  | ОТКЛОНЁН (Верно)
Bob/фраза_2              | Боб (чужак)          | 0.5288  | ОТКЛОНЁН (Верно)
Bob/фраза_3              | Боб (чужак)          | 0.5301  | ОТКЛОНЁН (Верно)
Bob/фраза_4              | Боб (чужак)          | 0.5283  | ОТКЛОНЁН (Верно)
Bob/фраза_5              | Боб (чужак)          | 0.5195  | ОТКЛОНЁН (Верно)
Charlie/фраза_1          | Чарли (чужак)        | 0.5696  | ОТКЛОНЁН (Верно)
Charlie/фраза_2          | Чарл

[+] Сохранён график распределения оценок: lab08_genuine_vs_impostor.png


[+] Сохранён график FAR/FRR/EER: lab08_far_frr_threshold.png


[+] Сохранён график влияния enrollment: lab08_enrollment_samples_effect.png

ИТОГИ ЗАНЯТИЯ 8:
1. Эталон Дархана Аканова сформирован из N=3 образцов.
2. Средний score своих (Genuine): 0.9562
3. Средний score чужих (Impostor): 0.5482
4. Динамический диапазон разделимости: Δ = 0.4080
5. Оптимальный порог: T* = 0.756 (EER = 0.0%)
Готово! Оптимальный порог T* = 0.756, EER = 0.0%
